In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect("food_waste_management.db")

In [2]:
# 1. How many food providers and receivers are there in each city?
df = pd.read_sql_query("""SELECT p.City,
               COUNT(DISTINCT p.Provider_ID) AS Total_Providers,
               COUNT(DISTINCT r.Receiver_ID) AS Total_Receivers
        FROM Providers p
        JOIN Receivers r ON p.City = r.City
        GROUP BY p.City
        LIMIT 8;""", conn)
df

,City,Total_Providers,Total_Receivers
0,Allenborough,1,1
1,Campbellchester,1,1
2,Christinaland,1,1
3,Christopherchester,1,2
4,Davidland,1,1
5,Davidport,1,1
6,East Ashleyshire,1,1
7,East Emily,1,1


In [3]:

# 2. Which type of food provider has provided the most total quantity of food?
df = pd.read_sql_query("""
    SELECT Provider_Type, SUM(Quantity) AS Total_Food_Quantity
    FROM Food_Listings
    GROUP BY Provider_Type
    ORDER BY Total_Food_Quantity DESC
    LIMIT 5;
""", conn)
df


,Provider_Type,Total_Food_Quantity
0,Restaurant,6923
1,Supermarket,6696
2,Catering Service,6116
3,Grocery Store,6059


In [4]:
# 3. Contact info of providers in a specific city
df = pd.read_sql_query("""SELECT Name, Contact, Address
        FROM Providers
        WHERE City = 'Jessestad';""", conn)
df

,Name,Contact,Address
0,Wright Ltd,850-175-4730,"69921 Sullivan Manor Apt. 996\nRussellport, ME..."


In [5]:
# 4. Which receivers have claimed the most food?
df = pd.read_sql_query("""SELECT r.Name, COUNT(*) AS Total_Claims
        FROM Claims c
        JOIN Receivers r ON c.Receiver_ID = r.Receiver_ID
        GROUP BY r.Name
        ORDER BY Total_Claims DESC
        LIMIT 5;""", conn)
df

,Name,Total_Claims
0,William Frederick,5
1,Scott Hunter,5
2,Matthew Webb,5
3,Anthony Garcia,5
4,Kristine Martin,4


In [6]:
# 5. Total quantity of food available from all providers
df = pd.read_sql_query("""SELECT SUM(Quantity) AS Total_Quantity FROM Food_Listings;""", conn)
df

,Total_Quantity
0,25794


In [7]:
# 6. Which city has the highest number of food listings?
df = pd.read_sql_query("""SELECT Location AS City, COUNT(*) AS Listings
        FROM Food_Listings
        GROUP BY Location
        ORDER BY Listings DESC
        LIMIT 1;""", conn)
df

,City,Listings
0,South Kathryn,6


In [8]:
# 7. Most commonly available food types
df = pd.read_sql_query("""SELECT Food_Type, COUNT(*) AS Count
        FROM Food_Listings
        GROUP BY Food_Type
        ORDER BY Count DESC
        LIMIT 5;""", conn)
df

,Food_Type,Count
0,Vegetarian,336
1,Vegan,334
2,Non-Vegetarian,330


In [9]:
# 8. Food listings expiring soon (within 3 days)
df = pd.read_sql_query("""SELECT * FROM Food_Listings
        WHERE DATE(Expiry_Date) <= DATE('now', '+3 days')
        LIMIT 5;""", conn)
df

,Food_ID,Food_Name,Quantity,Expiry_Date,Provider_ID,Provider_Type,Location,Food_Type,Meal_Type


In [10]:
# 9. Number of claims per food item
df = pd.read_sql_query("""SELECT f.Food_Name, COUNT(c.Claim_ID) AS Claim_Count
        FROM Claims c
        JOIN Food_Listings f ON c.Food_ID = f.Food_ID
        GROUP BY f.Food_Name
        ORDER BY Claim_Count DESC
        LIMIT 5;""", conn)
df

,Food_Name,Claim_Count
0,Rice,122
1,Soup,114
2,Dairy,110
3,Fish,108
4,Salad,106


In [11]:
# 10. Provider with highest number of successful claims
df = pd.read_sql_query("""SELECT p.Name, COUNT(*) AS Success_Count
        FROM Claims c
        JOIN Food_Listings f ON c.Food_ID = f.Food_ID
        JOIN Providers p ON f.Provider_ID = p.Provider_ID
        WHERE c.Status = 'Completed'
        GROUP BY p.Name
        ORDER BY Success_Count DESC
        LIMIT 5;""", conn)
df

,Name,Success_Count
0,Barry Group,5
1,Miller Inc,4
2,"Harper, Blake and Alexander",4
3,Butler-Richardson,4
4,"Barnes, Castro and Curtis",4


In [12]:
# 11. City with fastest claim rate
df = pd.read_sql_query("""SELECT f.Location AS City,
               AVG(JULIANDAY(c.Timestamp) - JULIANDAY(f.Expiry_Date)) AS Avg_Claim_Time
        FROM Food_Listings f
        JOIN Claims c ON f.Food_ID = c.Food_ID
        GROUP BY f.Location
        ORDER BY Avg_Claim_Time
        LIMIT 5;""", conn)
df

,City,Avg_Claim_Time
0,Adambury,None
1,Alexanderchester,None
2,Allenborough,None
3,Allenton,None
4,Amandashire,None


In [13]:
# 12. Percentage and count of claims by status
df = pd.read_sql_query("""SELECT Status,
               COUNT(*) AS Count,
               ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM Claims), 2) AS Percentage
        FROM Claims
        GROUP BY Status;""", conn)
df

,Status,Count,Percentage
0,Cancelled,336,33.6
1,Completed,339,33.9
2,Pending,325,32.5


In [14]:
# 13. Average quantity claimed per receiver
df = pd.read_sql_query("""SELECT r.Name, ROUND(AVG(f.Quantity), 2) AS Avg_Quantity
        FROM Claims c
        JOIN Food_Listings f ON c.Food_ID = f.Food_ID
        JOIN Receivers r ON c.Receiver_ID = r.Receiver_ID
        GROUP BY r.Name
        Order BY Avg_Quantity DESC
        LIMIT 5;""", conn)
df

,Name,Avg_Quantity
0,Thomas Villanueva,50.0
1,Peggy Knight,50.0
2,Nancy Silva,50.0
3,Nancy Jones,50.0
4,Lisa Pitts,50.0


In [15]:
# 14. Most claimed meal type
df = pd.read_sql_query("""SELECT Meal_Type, COUNT(*) AS Total_Claims
        FROM Food_Listings f
        JOIN Claims c ON f.Food_ID = c.Food_ID
        GROUP BY Meal_Type
        ORDER BY Total_Claims DESC
        LIMIT 5;""", conn)
df

,Meal_Type,Total_Claims
0,Breakfast,278
1,Lunch,250
2,Snacks,240
3,Dinner,232


In [16]:
# 15. Total quantity of food donated by each provider
df = pd.read_sql_query("""SELECT p.Name, SUM(f.Quantity) AS Total_Quantity
        FROM Food_Listings f
        JOIN Providers p ON f.Provider_ID = p.Provider_ID
        GROUP BY p.Name
        ORDER BY Total_Quantity DESC
        LIMIT 5;""", conn)
df

,Name,Total_Quantity
0,Miller Inc,217
1,Barry Group,179
2,"Evans, Wright and Mitchell",158
3,Smith Group,150
4,Campbell LLC,145


In [17]:

# 16. Top 5 cities by food claim percentage (fixed)
df = pd.read_sql_query("""
    SELECT Location, 
           ROUND(COUNT(Claims.Claim_ID) * 100.0 / COUNT(Food_Listings.Food_ID), 2) AS Claim_Percentage
    FROM Food_Listings
    LEFT JOIN Claims ON Food_Listings.Food_ID = Claims.Food_ID
    GROUP BY Location
    ORDER BY Claim_Percentage DESC
    LIMIT 5;
""", conn)
df


,Location,Claim_Percentage
0,Youngchester,100.0
1,Yatesside,100.0
2,Woodport,100.0
3,Wilsonview,100.0
4,Wilsonport,100.0


In [18]:

# 17. Food items never claimed (grouped by food and quantity)
df = pd.read_sql_query("""
    SELECT Food_Name, Provider_ID, Expiry_Date
    FROM Food_Listings
    WHERE Food_ID NOT IN (
        SELECT DISTINCT Food_ID FROM Claims
    )
    ORDER BY Expiry_Date ASC
    LIMIT 10;""", conn)
df


,Food_Name,Provider_ID,Expiry_Date
0,Fruits,930,3/16/2025
1,Rice,179,3/16/2025
2,Rice,150,3/16/2025
3,Fruits,174,3/16/2025
4,Pasta,905,3/16/2025
5,Chicken,967,3/16/2025
6,Dairy,391,3/16/2025
7,Salad,826,3/16/2025
8,Fruits,523,3/16/2025
9,Fruits,783,3/16/2025


In [19]:
# 18. Meal types donated most in each city
df = pd.read_sql_query("""SELECT Location AS City, Meal_Type, COUNT(*) AS Count
        FROM Food_Listings
        GROUP BY Location, Meal_Type
        ORDER BY City, Count DESC
        LIMIT 5;""", conn)
df

,City,Meal_Type,Count
0,Adambury,Lunch,1
1,Alexanderchester,Dinner,1
2,Allenborough,Dinner,1
3,Allenton,Snacks,1
4,Allenton,Lunch,1


In [20]:
# 19. Most claimed food type in each city
df = pd.read_sql_query("""SELECT f.Location AS City, f.Food_Type, COUNT(*) AS Count
        FROM Claims c
        JOIN Food_Listings f ON c.Food_ID = f.Food_ID
        GROUP BY f.Location, f.Food_Type
        ORDER BY Count DESC
        LIMIT 5;""", conn)
df

,City,Food_Type,Count
0,East Heatherport,Non-Vegetarian,8
1,New Carol,Non-Vegetarian,6
2,East Austin,Vegan,5
3,East John,Vegetarian,5
4,Jamesfurt,Vegan,5


In [21]:
# 20. Provider with the lowest claim rate
df = pd.read_sql_query("""SELECT p.Name,
               ROUND(COUNT(c.Claim_ID) * 1.0 / COUNT(DISTINCT f.Food_ID), 2) AS Claim_Rate
        FROM Food_Listings f
        LEFT JOIN Claims c ON f.Food_ID = c.Food_ID
        JOIN Providers p ON f.Provider_ID = p.Provider_ID
        GROUP BY p.Name
        ORDER BY Claim_Rate ASC
        LIMIT 5;""", conn)
df

,Name,Claim_Rate
0,Abbott-Robinson,0.0
1,Adams-Meyer,0.0
2,Adams-Young,0.0
3,Aguilar Group,0.0
4,Allen LLC,0.0


In [22]:

# 21. Overall claim success rate for all listings
df = pd.read_sql_query("""
    SELECT 
        ROUND(COUNT(DISTINCT Claims.Food_ID) * 100.0 / COUNT(DISTINCT Food_Listings.Food_ID), 2) AS Claim_Success_Rate
    FROM Food_Listings
    LEFT JOIN Claims ON Food_Listings.Food_ID = Claims.Food_ID;
""", conn)
df

,Claim_Success_Rate
0,64.7


In [23]:
conn.close()